# Track 3: 257 fresh tasks, and merged weights a judge can load

`sft4` scores 93.33% (84/90) on the frozen hard-90 holdout. That is 90 tasks, so the Wilson interval is 86.2-96.9% - the widest one in the entry. This kernel grades the same adapter on **257 further tasks** from the same six families, built from fresh seeds and deduplicated against every split the model ever trained on or was graded on (`data/build_hard257.py`, seed `20261113`, pinned by `data/hard257_eval.sha256`). Per family: 50 settle5, 50 two_query, 50 trap_combo, 50 paraphrase, 50 convert_sum, and 7 empty_balance (its pool of phrasings x names is exhausted, so the cap is declared instead of hidden).

It also publishes the adapter **merged into the base weights**, so the shipped behaviour loads with `transformers` alone, and checks that the merged copy reproduces the adapter's verdicts.

| step | time on one T4 (wall) | quota |
|---|---:|---:|
| setup, install, 29 offline checks | 5 min | 0.2 h |
| merge the published adapter | 3 min | 0.1 h |
| hard-257, `sft4` | ~3.6 h | 7.2 h |
| merged-vs-adapter parity on 3 tasks | 10 min | 0.3 h |
| upload run + merged weights | 10 min | 0.3 h |
| **total** | **~4 h** | **~8 h** |

Kaggle bills a T4 x2 machine as two GPU-hours per hour, and every cell here pins one GPU, so the machine is deliberately the single-GPU shape. If the session is stopped, re-run only the eval cell with `--offset N` where N is the number of `[i/257]` lines already printed; shards merge with `scripts/merge_runs.py`.

Optional last cell: the base model on the same 257 tasks (another ~5.4 h wall, 10.8 h quota) for a before/after column on this split. The paired base -> adapter test already exists on hard-90 (81 fixed, 0 broken, p < 1e-20), so this cell is nice-to-have, not needed for the claim.

In [ ]:
import os
from pathlib import Path

from kaggle_secrets import UserSecretsClient

try:
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    raise SystemExit(
        "HF_TOKEN is not attached. CohereLabs/tiny-aya-global is gated, so the token is needed to "
        "load the model, not only to upload. Add-ons -> Secrets, then Save & Run All."
    ) from exc

os.environ["HF_TOKEN"] = token
env = Path("/tmp/hf_env")
env.write_text('export HF_TOKEN="' + token + '"\n', encoding="utf-8")
env.chmod(0o600)
print("HF_TOKEN set (not printed)")

!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /kaggle/working | tail -1

## Setup: clone, install, run the offline suite

The checks are CPU-only and pin every frozen split by SHA-256, including the new hard-257 file, so a changed task file fails here instead of quietly moving the score.

In [ ]:
%%bash
set -euo pipefail
source /tmp/hf_env
cd /kaggle/working
rm -rf lexsi-agentic-hinglish
git clone -q https://github.com/PRINCE2-AI/lexsi-agentic-hinglish.git
cd lexsi-agentic-hinglish
git log --oneline -1
python -m pip install -q -r requirements.txt
python scripts/verify_all.py

## Merge the published adapter

Same weights the headline uses, `merge_and_unload()` applied, saved to `/kaggle/working/sft4_merged`. Skipped if the folder already exists.

In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
cd /kaggle/working/lexsi-agentic-hinglish
if [ ! -f /kaggle/working/sft4_merged/config.json ]; then
python - <<'PY'
import torch
from huggingface_hub import snapshot_download
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base = "CohereLabs/tiny-aya-global"
repo = snapshot_download("primce/tiny-aya-hinglish-lora", allow_patterns=["sft4_adapter/*"],
                         local_dir="/kaggle/working/sft4_repo")
tokenizer = AutoTokenizer.from_pretrained(base)
model = AutoModelForCausalLM.from_pretrained(base, dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(model, repo, subfolder="sft4_adapter")
merged = model.merge_and_unload()
merged.save_pretrained("/kaggle/working/sft4_merged", safe_serialization=True)
tokenizer.save_pretrained("/kaggle/working/sft4_merged")
print("merged -> /kaggle/working/sft4_merged")
PY
fi
python - <<'PY'
from huggingface_hub import snapshot_download
snapshot_download("primce/tiny-aya-hinglish-lora", allow_patterns=["sft4_adapter/*"],
                  local_dir="/kaggle/working/lexsi-agentic-hinglish/runs")
print("adapter -> runs/sft4_adapter")
PY
ls -d /kaggle/working/sft4_merged runs/sft4_adapter
df -h /kaggle/working | tail -1

## hard-257, shipped adapter

fp16, greedy, no 4-bit - the same settings as every published run, so the two splits are comparable. Expect ~50 s per task, so ~3.6 h. Results stream to `runs/sft4_hard257/` one line per task; if the session stops, note the last index and re-run with `--offset` set to it.

In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
export CUDA_VISIBLE_DEVICES=0
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
cd /kaggle/working/lexsi-agentic-hinglish
if [ ! -f /kaggle/working/sft4_merged/config.json ]; then echo "MERGE MISSING"; exit 0; fi

time python scripts/run_baseline.py --model /kaggle/working/sft4_merged \
  --tasks data/tasks_hard257_eval.jsonl --no-4bit --out runs/sft4_hard257 \
  || echo "HARD257 FAILED"

python scripts/job_upload.py --run sft4_hard257 || echo "UPLOAD FAILED"

## Merged weights: do they behave like the adapter?

Three tasks through both copies - the adapter on the base model, then the merged checkpoint - and a line-by-line comparison of the verdicts. Any difference means the merge is not the shipped model.

In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
export CUDA_VISIBLE_DEVICES=0
cd /kaggle/working/lexsi-agentic-hinglish
if [ ! -f /kaggle/working/sft4_merged/config.json ]; then echo "MERGE MISSING"; exit 0; fi

python scripts/run_baseline.py --model CohereLabs/tiny-aya-global --adapter runs/sft4_adapter \
  --tasks data/tasks_hard257_eval.jsonl --limit 3 --no-4bit --out runs/adapter_parity \
  || echo "PARITY ADAPTER FAILED"

python scripts/run_baseline.py --model /kaggle/working/sft4_merged \
  --tasks data/tasks_hard257_eval.jsonl --limit 3 --no-4bit --out runs/merged_parity \
  || echo "PARITY MERGED FAILED"

python - <<'PY'
import json
from pathlib import Path

def rows(name):
    path = Path("runs") / name / "results.jsonl"
    if not path.exists():
        return None
    return {r["task_id"]: r for r in (json.loads(l) for l in path.open(encoding="utf-8") if l.strip())}

a, m = rows("adapter_parity"), rows("merged_parity")
if not a or not m:
    print("parity: a side is missing, nothing to compare")
else:
    same = True
    for task_id in sorted(set(a) | set(m)):
        left, right = a.get(task_id), m.get(task_id)
        keys = ("success", "calls", "strict")
        equal = bool(left and right and all(left.get(k) == right.get(k) for k in keys))
        same = same and equal
        print(task_id, "MATCH" if equal else "DIFF",
              {k: (left or {}).get(k) for k in keys}, {k: (right or {}).get(k) for k in keys})
    print("merged == adapter on all three tasks:", same)
PY

python scripts/job_upload.py --run merged_parity --run adapter_parity || echo "PARITY UPLOAD FAILED"

## Publish the merged checkpoint

New model repository `primce/tiny-aya-hinglish-merged`, `README.md` taken from `docs/MERGED_CARD.md`, so the card and the weights are versioned together in git.

In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
cd /kaggle/working/lexsi-agentic-hinglish
if [ ! -f /kaggle/working/sft4_merged/config.json ]; then echo "MERGE MISSING"; exit 0; fi

python - <<'PY'
from huggingface_hub import HfApi, create_repo

repo = "primce/tiny-aya-hinglish-merged"
create_repo(repo, repo_type="model", exist_ok=True)
api = HfApi()
api.upload_folder(folder_path="/kaggle/working/sft4_merged", repo_id=repo,
                  ignore_patterns=["*.bin", "*.pt"])
api.upload_file(path_or_fileobj="docs/MERGED_CARD.md", path_in_repo="README.md", repo_id=repo)
print("uploaded", repo)
PY

## Read the result

Prints the hard-257 row, the per-family split and the solved count, so the new number can be compared with hard-90 without re-deriving anything by hand.

In [ ]:
import json
from pathlib import Path

ROOT = Path("/kaggle/working/lexsi-agentic-hinglish")
run = ROOT / "runs" / "sft4_hard257"
if not (run / "summary.json").exists():
    print("sft4_hard257: MISSING (eval cell did not finish)")
else:
    s = json.loads((run / "summary.json").read_text(encoding="utf-8"))
    print({k: s.get(k) for k in ("n", "success", "tac", "arr", "calls", "ter", "lcf",
                                 "strict_rate", "answer_ok", "seconds")})
    print("solved:", round((s["success"] or 0) * (s["n"] or 0)), "of", s["n"])
    print()
    families = {t["id"]: t["family"] for t in
                (json.loads(l) for l in (ROOT / "data" / "tasks_hard257_eval.jsonl").open(encoding="utf-8") if l.strip())}
    rows = [json.loads(l) for l in (run / "results.jsonl").open(encoding="utf-8") if l.strip()]
    per = {}
    for r in rows:
        fam = families.get(r["task_id"], "?")
        got, total = per.get(fam, (0, 0))
        per[fam] = (got + int(bool(r["success"])), total + 1)
    print("per family:")
    for fam in sorted(per):
        got, total = per[fam]
        print(f"  {fam:14s} {got:3d}/{total:<3d} {100 * got / total:5.1f}%")

## Optional: base model on the same 257 tasks

Only if the session has 5.5 h or more left. It gives a before/after column on this split; the paired test against base is already published on hard-90. Skip it and the claim is unaffected.

In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
export CUDA_VISIBLE_DEVICES=0
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
cd /kaggle/working/lexsi-agentic-hinglish
time python scripts/run_baseline.py --model CohereLabs/tiny-aya-global \
  --tasks data/tasks_hard257_eval.jsonl --no-4bit --out runs/base_hard257 \
  || echo "BASE HARD257 FAILED"
python scripts/job_upload.py --run base_hard257 || echo "BASE UPLOAD FAILED"

## What to paste back

Only the printed rows matter:

- `sft4_hard257.success` with `n` = 257 and the solved count - the new headline row.
- the per-family table - specifically whether `two_query` and `convert_sum`, the two families that fail on hard-90, hold the same rate when they have 50 graded tasks instead of 15.
- `merged == adapter on all three tasks: True` - the merged checkpoint is the shipped model.
- `seconds`, so the time table in `docs/RESULTS.md` stays real.
- if you ran the optional cell: `base_hard257.success`.